# Qdrant Product Search Pipeline
**Step by step: CSV → Preprocess → Embed → Store → Retrieve → Evaluate**

## Step 1: Install & Import

In [1]:
# install if needed
# !pip install pandas sentence-transformers qdrant-client

import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

print('All imports done')

All imports done


## Step 2: Load CSV
Load product data from Excel/CSV file

In [2]:
df = pd.read_csv('products.csv')

print(f'Total products: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(5)

Total products: 50
Columns: ['Product Name', 'Barcode', 'Features', 'Description', 'Tags']



,Product Name,Barcode,Features,Description,Tags
0,Basmati Rice,BR001,Long grain aromatic aged,Premium quality long grain basmati rice aged f...,rice grain food staple biryani pulao aromatic ...
1,Brown Rice,BR002,Whole grain high fiber nutritious,Healthy whole grain brown rice rich in dietary...,rice grain healthy fiber nutritious wholesome ...
2,White Rice,BR003,Short grain soft fluffy daily use,Regular short grain white rice perfect for dai...,rice grain food staple soft fluffy daily cooki...
3,Tata Salt,TS001,Iodized fine grain refined,Iodized fine grain salt for daily cooking and ...,salt iodized cooking kitchen mineral health es...
4,Rock Salt,RS001,Natural mineral rich unrefined,Natural rock salt with essential minerals mine...,salt natural mineral ayurvedic cooking pink hi...


## Step 3: Preprocess
- Combine all columns into one text
- Lowercase
- Clean spaces
- Handle missing values

**Note: DO NOT remove stopwords for transformer embeddings**

In [3]:
def preprocess(row):
    # combine all fields
    fields = [
        str(row.get('Product Name', '')),
        str(row.get('Features', '')),
        str(row.get('Description', '')),
        str(row.get('Tags', ''))
    ]
    # remove NaN values
    fields = [f for f in fields if f != 'nan']
    
    # combine
    text = ' '.join(fields)
    
    # lowercase
    text = text.lower()
    
    # remove extra spaces
    text = ' '.join(text.split())
    
    return text

df['combined_text'] = df.apply(preprocess, axis=1)

print('Sample combined text:')
print(df['combined_text'][0])
print()

print(f'Text length range: {df["combined_text"].str.len().min()} - {df["combined_text"].str.len().max()} chars')

Sample combined text:
basmati rice long grain aromatic aged premium quality long grain basmati rice aged for extra aroma. perfect for biryani pulao and fried rice. each grain stays separate after cooking. sourced from himalayan foothills with rich soil. ideal for special occasions and daily cooking. rice grain food staple biryani pulao aromatic aged long grain

Text length range: 313 - 439 chars


## Step 4: Generate Embeddings
Convert each product text → 384-dimensional vector using all-MiniLM-L6-v2

In [4]:
# load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded')
print(f'Embedding dimensions: {model.get_sentence_embedding_dimension()}')

# generate embeddings for all products
embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'  {embeddings.shape[0]} products')
print(f'  {embeddings.shape[1]} dimensions per product')
print(f'\nSample vector (first 5 values of product 0):')
print(embeddings[0][:5])

/home/digit7/shanmugamani/nahs/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:241: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded
Embedding dimensions: 384


/tmp/ipykernel_301103/3870831708.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Embedding dimensions: {model.get_sentence_embedding_dimension()}')


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Embeddings shape: (50, 384)
  50 products
  384 dimensions per product

Sample vector (first 5 values of product 0):
[ 0.01500181  0.02723848  0.04675621 -0.00363085 -0.02764251]


## Step 5: Store in Qdrant
- Create collection with 384 dimensions
- Each product = one Point (vector + payload)
- Payload stores: name, barcode, features, tags

In [5]:
# connect to Qdrant (in-memory for testing)
client = QdrantClient(':memory:')

# create collection
client.create_collection(
    collection_name='products',
    vectors_config=VectorParams(
        size=384,              # embedding dimensions
        distance=Distance.COSINE  # similarity measure
    )
)
print('Collection created')

# create points
points = []
for idx, row in df.iterrows():
    points.append(PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload={
            'name':        row['Product Name'],
            'barcode':     str(row['Barcode']),
            'features':    str(row['Features']),
            'tags':        str(row['Tags'])
        }
    ))

# store in Qdrant
client.upsert(collection_name='products', points=points)
print(f'Stored {len(points)} products in Qdrant')

# verify
info = client.get_collection('products')
print(f'Collection points count: {info.points_count}')

Collection created
Stored 50 products in Qdrant
Collection points count: 50


## Step 6: Retrieve
Search products using natural language queries

In [10]:
def search(query, top_k=3, threshold=0.3):
    # convert query to vector
    query_vector = model.encode(query).tolist()
    
    # search Qdrant
    results = client.query_points(
        collection_name='products',
        query=query_vector,
        limit=top_k,
        score_threshold=threshold
    ).points
    
    return results

# test queries
queries = [
    'long grain rice',
]

for query in queries:
    print(f"Query: '{query}'")
    results = search(query)
    print(results)
    
    for r in results:
        print(f"  [{r.score:.3f}] {r.payload['name']} ({r.payload['barcode']})")
    if not results:
        print('  No results found')
    print()

Query: 'long grain rice'
[ScoredPoint(id=1, version=0, score=0.6440649906225305, payload={'name': 'Brown Rice', 'barcode': 'BR002', 'features': 'Whole grain high fiber nutritious', 'tags': 'rice grain healthy fiber nutritious wholesome diabetic diet weight loss'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=2, version=0, score=0.6303361053839407, payload={'name': 'White Rice', 'barcode': 'BR003', 'features': 'Short grain soft fluffy daily use', 'tags': 'rice grain food staple soft fluffy daily cooking south indian'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=0, version=0, score=0.5758991269214575, payload={'name': 'Basmati Rice', 'barcode': 'BR001', 'features': 'Long grain aromatic aged', 'tags': 'rice grain food staple biryani pulao aromatic aged long grain'}, vector=None, shard_key=None, order_value=None)]
  [0.644] Brown Rice (BR002)
  [0.630] White Rice (BR003)
  [0.576] Basmati Rice (BR001)



## Step 7: Evaluate Retrieval Quality
- **Hit Rate**: Did we get at least 1 correct result?
- **Precision@K**: How many returned results are correct?
- **Ground truth**: What SHOULD be returned

In [7]:
# ground truth
ground_truth = {
    'basmati rice':               ['BR001'],
    'something healthy for breakfast': ['GT001', 'MA001'],
    'cleaning product for bathroom':   ['HC001'],
    'snack for evening':             ['LC001', 'KK001'],
    'dairy product protein':         ['EF001', 'PF001'],
    'writing stationery':            ['NB001', 'PR001'],
}

hit_count      = 0
precision_list = []
total          = len(ground_truth)

print(f'{"─"*60}')
for query, expected in ground_truth.items():
    results   = search(query, top_k=3)
    retrieved = [r.payload['barcode'] for r in results]
    
    # hit rate
    hit = any(b in retrieved for b in expected)
    if hit:
        hit_count += 1
    
    # precision@3 — how many of top 3 are correct
    correct    = sum(1 for b in retrieved if b in expected)
    precision  = correct / len(retrieved) if retrieved else 0
    precision_list.append(precision)
    
    status = '✓ HIT ' if hit else '✗ MISS'
    print(f'{status} | P@3={precision:.2f} | {query}')
    print(f'         Expected : {expected}')
    print(f'         Retrieved: {retrieved}')
    print()

print(f'{"─"*60}')
print(f'Hit Rate    : {hit_count}/{total} = {hit_count/total:.0%}')
print(f'Avg P@3     : {sum(precision_list)/len(precision_list):.2f}')

────────────────────────────────────────────────────────────
✓ HIT  | P@3=0.33 | basmati rice
         Expected : ['BR001']
         Retrieved: ['BR001', 'BR003', 'BR002']

✓ HIT  | P@3=0.33 | something healthy for breakfast
         Expected : ['GT001', 'MA001']
         Retrieved: ['BWB001', 'EF001', 'GT001']

✓ HIT  | P@3=0.33 | cleaning product for bathroom
         Expected : ['HC001']
         Retrieved: ['HC001', 'SE001', 'DS001']

✓ HIT  | P@3=0.67 | snack for evening
         Expected : ['LC001', 'KK001']
         Retrieved: ['LC001', 'BG001', 'KK001']

✓ HIT  | P@3=0.67 | dairy product protein
         Expected : ['EF001', 'PF001']
         Retrieved: ['MA001', 'PF001', 'EF001']

✓ HIT  | P@3=0.67 | writing stationery
         Expected : ['NB001', 'PR001']
         Retrieved: ['PR001', 'NB001', 'AP001']

────────────────────────────────────────────────────────────
Hit Rate    : 6/6 = 100%
Avg P@3     : 0.50


## Step 8: Improve — Try different thresholds

In [8]:
print('Threshold experiment:')
print(f'{"─"*40}')

queries_gt = {
    'long grain rice':               ['BR001'],
    'something healthy for breakfast': ['GT001', 'MA001'],
    'cleaning product for bathroom':   ['HC001'],
    'snack for evening':             ['LC001', 'KK001'],
    'dairy product protein':         ['EF001', 'PF001'],
    'writing stationery':            ['NB001', 'PR001'],
}
total = len(queries_gt)

for threshold in [0.7, 0.5, 0.4, 0.3, 0.2]:
    hits = 0
    for query, expected in queries_gt.items():
        results   = search(query, top_k=3, threshold=threshold)
        retrieved = [r.payload['barcode'] for r in results]
        if any(b in retrieved for b in expected):
            hits += 1
    print(f'Threshold={threshold} → Hit Rate: {hits}/{total} = {hits/total:.0%}')

Threshold experiment:
────────────────────────────────────────
Threshold=0.7 → Hit Rate: 0/6 = 0%
Threshold=0.5 → Hit Rate: 4/6 = 67%
Threshold=0.4 → Hit Rate: 6/6 = 100%
Threshold=0.3 → Hit Rate: 6/6 = 100%
Threshold=0.2 → Hit Rate: 6/6 = 100%


## Step 9: Visualize Vectors (PCA)
Reduce 384 dimensions → 2D and plot to see how similar products cluster together

## Step 10: Visualize Vectors (t-SNE)
t-SNE gives better clusters than PCA — slower but more accurate visualization

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# category → barcodes mapping
categories = {
    'Grain/Rice':  ['BR001','BR002','BR003','WF001','MF001'],
    'Fruit':       ['AF001','BN001','MN001','OR001','GR001'],
    'Vegetable':   ['TM001','ON001','PT001','CR001','SP001'],
    'Dairy/Meat':  ['MA001','CA001','BA001','PF001','EF001','CB001','ML001'],
    'Snack':       ['LC001','KK001','BG001','BB001','BWB001'],
    'Oil/Cooking': ['SO001','CO001','OO001','TS001','RS001'],
    'Hygiene':     ['DS001','DV001','HS001','CT001','VD001','SE001','HC001'],
    'Beverage':    ['TT001','GT001','NC001'],
    'Other':       ['NB001','PR001','AP001','WB001','UM001','TE001','MG001','PP001'],
}
colors = {
    'Grain/Rice': 'brown', 'Fruit': 'green', 'Vegetable': 'lime',
    'Dairy/Meat': 'blue',  'Snack': 'orange','Oil/Cooking': 'gold',
    'Hygiene': 'purple',   'Beverage': 'cyan','Other': 'gray',
}

def plot_vectors(reduced, title, filename):
    plt.figure(figsize=(14, 10))
    for category, barcodes in categories.items():
        idx = [df[df['Barcode'] == b].index[0] for b in barcodes if b in df['Barcode'].values]
        if not idx: continue
        plt.scatter(reduced[idx, 0], reduced[idx, 1],
                    label=category, color=colors[category], s=120, alpha=0.8)
        for i in idx:
            plt.annotate(df['Product Name'][i], (reduced[i, 0], reduced[i, 1]),
                         fontsize=7, ha='right', va='bottom')
    plt.title(title, fontsize=13)
    plt.legend(loc='upper right', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f'Saved: {filename}')

# ── PCA ───────────────────────────────────────────────────
pca          = PCA(n_components=2)
reduced_pca  = pca.fit_transform(embeddings)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}')
plot_vectors(reduced_pca, 'Product Vectors — PCA\nSimilar products cluster together', 'qdrant_pca.png')

# ── t-SNE ─────────────────────────────────────────────────
print('\nRunning t-SNE (slower but better clusters)...')
tsne         = TSNE(n_components=2, random_state=42, perplexity=10, max_iter=1000)
reduced_tsne = tsne.fit_transform(embeddings)
plot_vectors(reduced_tsne, 'Product Vectors — t-SNE\nBetter clusters than PCA', 'qdrant_tsne.png')